# 2D 軸対称ストリーマーの格子依存性

**目的**：2D 軸対称の計算結果（速度・先端の電界・半径）が、格子の細かさでどれだけ変わるかを確かめる。

* 元の設定：`programs/standard_2d/streamer_cyl.cfg`（空気、背景電界 2 MV/m、光電離なし）
* 変えるもの：`refine_adx`（$\alpha\,\Delta x$ がこの値より大きいセルを細分化する。小さいほど格子が細かい）
  を 2, 1（標準）, 0.5, 0.25 の 4 段階
* 計算時間：5 ns（速度は 1 ns 以降ほぼ一定）

**比べる量**

| 量 | ログの列 | 意味 |
|---|---|---|
| 速度 | `ne_zmin` | 先端位置（電子密度がしきい値を超える最小の z）の傾き |
| 先端の電界 | `max(E)` | 最大電界（先端にある） |
| 半径 | `max(E_r)` の r 座標 | 径方向の電界が最大になる位置（電気力学的な半径） |

計算済みの結果があれば再計算しません（`reuse=True`）。
`adx = 0.25` は 1.5 時間以上かかるので、`RUN_FINEST = True` にしたときだけ計算します。
（`False` のままでも、計算済みの結果があれば読み込んで比較に加えます。）

In [ ]:
import matplotlib.pyplot as plt
import matplotlib_fontja  # 日本語フォント
import numpy as np
import pandas as pd

from afivo_helpers import (REPO, run_streamer, load_log, lineout_settings,
                           load_lineout, front_velocity, silo_files, load_silo)

L2D = 16e-3
T_END = 5e-9
ADX_ALL = [2.0, 1.0, 0.5, 0.25]
FINEST = 0.25
RUN_FINEST = False      # True にすると adx = 0.25 も計算する（1.5 時間以上）

## 1. 計算（計算済みなら読み込むだけ）

In [ ]:
def finished(adx):
    log = REPO_OUT / f"grid_adx{adx}_log.txt"
    return log.exists() and load_log(str(log)[:-8])["time"].iloc[-1] >= 0.999 * T_END

REPO_OUT = REPO / "programs" / "standard_2d" / "output"
runs = {}
for adx in ADX_ALL:
    if adx == FINEST and not RUN_FINEST and not finished(adx):
        print(f"adx = {adx} は計算済みの結果がないので省略します")
        continue
    base = run_streamer("standard_2d", "streamer_cyl.cfg", f"grid_adx{adx}",
                        {"end_time": T_END, "refine_adx": adx, **lineout_settings(2)},
                        threads=4, show_every=20, reuse=True)
    runs[adx] = (base, load_log(base))
ADX = list(runs)
print("比較する refine_adx:", ADX)

## 2. 結果の一覧

In [ ]:
rows = []
for adx, (base, log) in runs.items():
    late = log["time"] >= 2e-9          # 立ち上がりの後（2〜5 ns）の平均
    rows.append({
        "refine_adx": adx,
        "速度 (km/s)": front_velocity(log["time"], log["ne_zmin"], 1e-3, L2D - 1e-3,
                                      t_min=1e-9) / 1e3,
        "先端電界 (MV/m)": log["max(E)"][late].mean() / 1e6,
        "半径 (mm)": log["x.2"][late].mean() * 1e3,
        "5 ns の先端位置 (mm)": log["ne_zmin"].iloc[-1] * 1e3,
        "最小格子幅 (μm)": log["min(dx)"].min() * 1e6,
        "セル数 (5 ns)": int(log["n_cells"].iloc[-1]),
        "計算時間 (s)": log["wc_time"].iloc[-1],
    })
table = pd.DataFrame(rows).set_index("refine_adx")
ref = table.loc[min(ADX)]                # いちばん細かい格子を基準にする
for c in ["速度 (km/s)", "先端電界 (MV/m)", "半径 (mm)"]:
    table[c.split(" ")[0] + " の差 (%)"] = 100 * (table[c] / ref[c] - 1)
table.round(3)

## 3. 時間変化

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.8), constrained_layout=True)
for adx, (base, log) in runs.items():
    t = log["time"][1:] * 1e9
    lbl = f"adx = {adx}"
    ax[0].plot(t, log["ne_zmin"][1:] * 1e3, label=lbl)
    ax[1].plot(t, log["max(E)"][1:] / 1e6, label=lbl)
    ax[2].plot(t, log["x.2"][1:] * 1e3, label=lbl)
ax[0].set_ylabel("先端位置 z (mm)")
ax[1].set_ylabel("最大電界 (MV/m)")
ax[2].set_ylabel("半径（max E_r の r, mm）")
for a in ax:
    a.set_xlabel("t (ns)"); a.grid(alpha=0.3); a.legend()

## 4. 先端付近の軸方向分布（5 ns）

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(9, 6), sharex=True, constrained_layout=True)
for adx, (base, log) in runs.items():
    df, t = load_lineout(base)
    ax[0].semilogy(df["y"] * 1e3, np.maximum(df["e"], 1e10), label=f"adx = {adx}")
    ax[1].plot(df["y"] * 1e3, df["electric_fld"] / 1e6, label=f"adx = {adx}")
z_tip = runs[min(ADX)][1]["ne_zmin"].iloc[-1] * 1e3
ax[1].set_xlim(z_tip - 1.5, z_tip + 2.5)
ax[0].set_ylim(1e13, 1e21)
ax[0].set_ylabel("電子密度 (m$^{-3}$)")
ax[1].set_ylabel("電界 (MV/m)"); ax[1].set_xlabel("z (mm)")
for a in ax:
    a.grid(alpha=0.3); a.legend()
fig.suptitle(f"r = 0 上の分布, t = {t*1e9:.1f} ns")

## 5. 電子密度の分布（5 ns）：粗い格子と細かい格子

In [ ]:
show = [max(ADX), 1.0, min(ADX)]
fig, axes = plt.subplots(1, len(show), figsize=(10, 5), constrained_layout=True)
for ax, adx in zip(axes, show):
    base = runs[adx][0]
    data, r, z, t = load_silo(silo_files(base)[-1], "e", r_max=[1.5e-3, 10e-3])
    im = ax.imshow(data.T, origin="lower", aspect="equal", cmap="inferno",
                   extent=[r[0]*1e3, r[-1]*1e3, z[0]*1e3, z[-1]*1e3], vmin=0, vmax=2e20)
    ax.set_title(f"adx = {adx}")
    ax.set_xlabel("r (mm)")
axes[0].set_ylabel("z (mm)")
fig.colorbar(im, ax=axes, shrink=0.8).set_label("電子密度 (m$^{-3}$)")

## 6. 読み方

* 計算した中でいちばん細かい格子を基準に、標準の `adx = 1` との差が数 % 以内なら、標準の格子で十分と言える。
* 差が大きい量があれば、その量を議論するときは格子を細かくする（計算時間とのかね合いで決める）。
* 1D の平面フロントでは、標準の格子だと 6 MV/m で速度が約 9% 速く出た（`compare_1d_2d.ipynb` の 5 節）。